# 09 — Pipeline estático con mapas de MediaPipe (~25.000 imágenes)

Este notebook entrena la rama estática y prepara la lógica de sesión para deletreo con cámara. En cada imagen, MediaPipe localiza la mano y los 21 puntos se convierten a un mapa RGB; MobileNetV2 clasifica las letras estáticas.

## Clases de este experimento

- Letras estáticas: `A–I` y `K–Y` (24 clases). `J` y `Z` quedan reservadas para la rama temporal de video.
- `space`: clase de control disponible en el dataset Kaggle y usada para insertar espacio en el texto.
- `nothing`: ausencia de mano / clase de rechazo.
- 960 imágenes por clase × 26 clases = **24.960 imágenes** (aprox. 25.000).

El modelo predice una clase por imagen. La cámara puede mostrar esa predicción siempre. Solo después de **Iniciar** se acumulan letras; al pulsar **Detener** se termina la sesión y se generan sugerencias de corrección. Las sugerencias ortográficas son una fase de texto posterior, no parte del clasificador visual.

La rama de J/Z se conectará cuando existan videos etiquetados. El notebook 07 contiene el formato y el preprocesamiento temporal iniciales.

In [ ]:
# Ejecuta en Google Colab. Selecciona GPU en Entorno de ejecución si tienes cuota disponible.
%pip install -q mediapipe==0.10.35 scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from IPython.display import display
import csv
import json
import random
import urllib.request
import warnings
import difflib

import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
SAMPLES_PER_CLASS = 960
IMAGE_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 6
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/proyecto-ia2")
RAW_DIR = DRIVE_ROOT / "data/raw/asl_alphabet_train/asl_alphabet_train"
OWN_PHOTOS_DIR = DRIVE_ROOT / "data/mis_fotos"
OUTPUT_DIR = DRIVE_ROOT / "data/processed/pipeline_mapas_25k"
MODEL_ASSET = OUTPUT_DIR / "hand_landmarker.task"

STATIC_LETTERS = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ" if c not in {"J", "Z"}]
CLASSES = STATIC_LETTERS + ["space", "nothing"]
CLASS_TO_ID = {label: i for i, label in enumerate(CLASSES)}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"No encuentro el dataset: {RAW_DIR}")
if not MODEL_ASSET.exists():
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_ASSET)

print(f"Clases ({len(CLASSES)}): {CLASSES}")
print(f"Muestra máxima: {SAMPLES_PER_CLASS * len(CLASSES):,} imágenes")
print("Salida:", OUTPUT_DIR)

## 1. Seleccionar datos y extraer mapas

Primero ejecuta el piloto de 5 imágenes por clase y revisa que MediaPipe detecte la mano y que el mapa conserve la configuración. Después se ejecutará la selección balanceada de 960 por clase. La extracción escribe un manifiesto por imagen y reanuda usando los mapas ya guardados.

In [ ]:
FINGER_COLORS = [(255,90,90),(255,210,40),(80,235,100),(50,190,255),(220,90,255)]
FINGER_EDGES = [[(0,1),(1,2),(2,3),(3,4)],[(0,5),(5,6),(6,7),(7,8)],[(0,9),(9,10),(10,11),(11,12)],[(0,13),(13,14),(14,15),(15,16)],[(0,17),(17,18),(18,19),(19,20)]]
PALM_EDGES = [(5,9),(9,13),(13,17)]
POINT_COLORS = {**{i:FINGER_COLORS[0] for i in range(1,5)}, **{i:FINGER_COLORS[1] for i in range(5,9)}, **{i:FINGER_COLORS[2] for i in range(9,13)}, **{i:FINGER_COLORS[3] for i in range(13,17)}, **{i:FINGER_COLORS[4] for i in range(17,21)}, 0:(255,255,255)}


def class_folder(root, label):
    return next((p for p in Path(root).iterdir() if p.is_dir() and p.name.strip().lower() == label.lower()), None)


def choose_images(root, per_class=SAMPLES_PER_CLASS, allow_missing=False):
    records = []
    for label in CLASSES:
        folder = class_folder(root, label)
        if folder is None:
            if allow_missing:
                continue
            raise FileNotFoundError(f"Falta la carpeta de clase {label!r} en {root}")
        paths = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        if per_class is not None and len(paths) > per_class:
            rng = np.random.default_rng(SEED + sum(map(ord, label)))
            chosen = np.sort(rng.choice(len(paths), size=per_class, replace=False))
            paths = [paths[i] for i in chosen]
        records.extend({"source": p, "source_rel": p.relative_to(root).as_posix(), "label": label} for p in paths)
    counts = pd.Series([r["label"] for r in records]).value_counts().reindex(CLASSES, fill_value=0)
    display(counts.rename("imágenes seleccionadas").to_frame())
    return records


def landmarks_to_map(hand_landmarks):
    xy = np.asarray([(p.x, p.y) for p in hand_landmarks], dtype=np.float32)
    if xy.shape != (21,2) or not np.isfinite(xy).all():
        return None
    xy -= xy[0]  # centra en la muñeca
    palm_scale = float(np.linalg.norm(xy[9]))
    if palm_scale < 1e-6:
        return None
    xy /= palm_scale
    extent = float(np.max(np.abs(xy)))
    if extent < 1e-6:
        return None
    xy = xy / extent * 0.40 * (IMAGE_SIZE-1) + (IMAGE_SIZE-1)/2
    points = [(int(round(x)),int(round(y))) for x,y in xy]
    canvas = Image.new("RGB", (IMAGE_SIZE,IMAGE_SIZE), (0,0,0))
    draw = ImageDraw.Draw(canvas)
    width, radius = max(2,IMAGE_SIZE//70), max(3,IMAGE_SIZE//48)
    for edges,color in zip(FINGER_EDGES,FINGER_COLORS):
        for a,b in edges:
            draw.line((points[a],points[b]), fill=color, width=width)
    for a,b in PALM_EDGES:
        draw.line((points[a],points[b]), fill=(235,235,235), width=width)
    for i,(x,y) in enumerate(points):
        draw.ellipse((x-radius,y-radius,x+radius,y+radius), fill=POINT_COLORS[i], outline=(255,255,255))
    return canvas


def create_detector():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


def prepare_maps(root, subset_name, per_class, progress_every=250, allow_missing=False):
    records = choose_images(root, per_class, allow_missing=allow_missing)
    out_root = OUTPUT_DIR / subset_name
    maps_root = out_root / "maps"
    manifest_path = out_root / "manifest.csv"
    maps_root.mkdir(parents=True, exist_ok=True)
    existing = {}
    if manifest_path.exists():
        old = pd.read_csv(manifest_path).drop_duplicates("source_rel", keep="last")
        existing = {str(row.source_rel): row._asdict() for row in old.itertuples(index=False)}
    header = not manifest_path.exists() or manifest_path.stat().st_size == 0
    detector = create_detector()
    try:
        with manifest_path.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=["source_rel","label","map_path","detected","error"])
            if header:
                writer.writeheader()
            for i, row in enumerate(records, 1):
                map_rel = f"{row['label']}/{Path(row['source_rel']).stem}_map.png"
                destination = maps_root / map_rel
                previous = existing.get(row["source_rel"])
                if previous and destination.is_file():
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                error = ""
                if row["label"] == "nothing":
                    mapped, detected = Image.new("RGB",(IMAGE_SIZE,IMAGE_SIZE)), False
                else:
                    try:
                        with Image.open(row["source"]) as opened:
                            image = ImageOps.exif_transpose(opened).convert("RGB")
                        rgb = np.asarray(image)
                        result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
                        if result.hand_landmarks:
                            mapped = landmarks_to_map(result.hand_landmarks[0])
                            detected = mapped is not None
                        else:
                            mapped, detected = None, False
                        if mapped is None:
                            mapped = Image.new("RGB",(IMAGE_SIZE,IMAGE_SIZE))
                    except Exception as exc:
                        mapped, detected = Image.new("RGB",(IMAGE_SIZE,IMAGE_SIZE)), False
                        error = f"{type(exc).__name__}: {exc}"
                mapped.save(destination, format="PNG")
                writer.writerow({"source_rel":row["source_rel"],"label":row["label"],"map_path":map_rel,"detected":int(detected),"error":error})
                if i % 100 == 0:
                    stream.flush()
                if i % progress_every == 0 or i == len(records):
                    stream.flush()
                    print(f"Procesadas {i:,}/{len(records):,} | detección de mano {detected}")
    finally:
        detector.close()
    return pd.read_csv(manifest_path).drop_duplicates("source_rel", keep="last")

# PASO 1: piloto visual: 5 imágenes de cada una de las 26 clases.
pilot_df = prepare_maps(RAW_DIR, "pilot", per_class=5)
print(f"Piloto: {len(pilot_df)} imágenes. Revisa los mapas en la siguiente celda.")

In [ ]:
# Revisa ejemplos detectados antes del lote grande.
shown = pilot_df[pilot_df.detected.astype(bool)].sample(
    min(12, int(pilot_df.detected.sum())), random_state=SEED
)
if shown.empty:
    print("MediaPipe no detectó manos en el piloto. Revisa la ruta y las imágenes antes de seguir.")
else:
    fig, axes = plt.subplots(3, 4, figsize=(10,8))
    for ax,row in zip(axes.flat, shown.itertuples(index=False)):
        ax.imshow(Image.open(OUTPUT_DIR / "pilot" / "maps" / row.map_path))
        ax.set_title(f"{row.label} | mano={bool(row.detected)}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(shown):]: ax.axis("off")
    plt.tight_layout(); plt.show()

## 2. Generar el subconjunto balanceado (~25.000)

Después de revisar el piloto, ejecuta esta celda. Toma 960 imágenes por clase (24.960 total). Si se interrumpe, vuelve a ejecutarla: los mapas que estén en el manifiesto se reutilizan.

In [ ]:
train_df = prepare_maps(RAW_DIR, "train_25k", per_class=SAMPLES_PER_CLASS, progress_every=250)
print(f"Dataset listo: {len(train_df):,} imágenes | mapas en {OUTPUT_DIR / 'train_25k' / 'maps'}")
print("Tasa de detección en letras y space:", f"{train_df[train_df.label != 'nothing'].detected.mean():.1%}")

## 3. Partición train/validación/test

Se guarda una única partición estratificada 70/15/15 para que el entrenamiento y los reportes sean reproducibles. Las señas donde MediaPipe falló se excluyen de entrenamiento/validación; en test permanecen y cuentan como errores del flujo completo.

In [ ]:
split_path = OUTPUT_DIR / "splits_25k.csv"
all_paths = set(train_df.source_rel.astype(str))
splits = pd.read_csv(split_path) if split_path.exists() else None
if splits is None or set(splits.source_rel.astype(str)) != all_paths:
    train_val,test = train_test_split(train_df,test_size=0.15,stratify=train_df.label,random_state=SEED)
    train,val = train_test_split(train_val,test_size=0.15/0.85,stratify=train_val.label,random_state=SEED)
    splits = pd.concat([train.assign(split="train"),val.assign(split="val"),test.assign(split="test")],ignore_index=True)
    splits.to_csv(split_path,index=False)
display(pd.crosstab(splits.split,splits.label))
train_rows = splits[(splits.split=="train") & ((splits.detected.astype(bool)) | (splits.label=="nothing"))].reset_index(drop=True)
val_rows = splits[(splits.split=="val") & ((splits.detected.astype(bool)) | (splits.label=="nothing"))].reset_index(drop=True)
test_rows = splits[splits.split=="test"].reset_index(drop=True)
print("Tamaños:",splits.split.value_counts().to_dict())
print(f"Usables train/val: {len(train_rows):,}/{len(val_rows):,}; test end-to-end: {len(test_rows):,}")

## 4. Entrenar MobileNetV2 sobre los mapas

La red recibe mapas RGB de 224×224. Se entrena primero la cabeza y luego se ajustan las últimas 30 capas. Los checkpoints de recuperación se escriben en Drive al final de cada época. Si Colab se desconecta, vuelve a ejecutar las celdas de configuración/funciones y luego la celda de entrenamiento, manteniendo la misma configuración.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, training=False):
    paths=[str(OUTPUT_DIR/"train_25k"/"maps"/p) for p in frame.map_path.astype(str)]
    labels=frame.label.map(CLASS_TO_ID).to_numpy(dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    if training: ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    def load(path,label):
        image=tf.io.decode_png(tf.io.read_file(path),channels=3)
        image=tf.image.resize(image,(IMAGE_SIZE,IMAGE_SIZE),antialias=True)
        return tf.cast(image,tf.float32),label
    ds=ds.map(load,num_parallel_calls=AUTOTUNE)
    if training:
        def augment(image,label):
            foreground=tf.reduce_max(image,axis=-1,keepdims=True)>0.5
            image=tf.image.random_contrast(image,0.85,1.15)
            image=tf.image.random_brightness(image,max_delta=15.0)
            image=tf.clip_by_value(image,0.0,255.0)
            return tf.where(foreground,image,tf.zeros_like(image)),label
        ds=ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_model():
    inputs=tf.keras.Input((IMAGE_SIZE,IMAGE_SIZE,3),name="landmark_map")
    x=tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    backbone=tf.keras.applications.MobileNetV2(include_top=False,weights="imagenet",input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
    backbone.trainable=False
    x=backbone(x,training=False)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.Dropout(0.30)(x)
    outputs=tf.keras.layers.Dense(len(CLASSES),activation="softmax",dtype="float32")(x)
    model=tf.keras.Model(inputs,outputs,name="mobilenetv2_landmark_maps")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model,backbone

def train_map_model():
    run_dir=OUTPUT_DIR/"model"
    run_dir.mkdir(parents=True,exist_ok=True)
    model,backbone=build_model()
    train_ds,val_ds=make_dataset(train_rows,True),make_dataset(val_rows)
    def callbacks_for(phase):
        return [
            tf.keras.callbacks.BackupAndRestore(backup_dir=str(run_dir/f"backup_{phase}")),
            tf.keras.callbacks.EarlyStopping(monitor="val_accuracy",mode="max",patience=2,restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=1,min_lr=1e-6),
            tf.keras.callbacks.ModelCheckpoint(str(run_dir/f"best_{phase}.keras"),monitor="val_accuracy",mode="max",save_best_only=True),
        ]
    head_history=model.fit(train_ds,validation_data=val_ds,epochs=HEAD_EPOCHS,callbacks=callbacks_for("head"),verbose=2)
    head_score=max(head_history.history.get("val_accuracy",[0.0]))
    head_path=run_dir/"head_model.keras"
    model.save(head_path)
    backbone.trainable=True
    for layer in backbone.layers[:-30]: layer.trainable=False
    for layer in backbone.layers:
        if isinstance(layer,tf.keras.layers.BatchNormalization): layer.trainable=False
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    fine_history=model.fit(train_ds,validation_data=val_ds,epochs=FINETUNE_EPOCHS,callbacks=callbacks_for("finetune"),verbose=2)
    fine_score=max(fine_history.history.get("val_accuracy",[0.0]))
    if fine_score<head_score: model=tf.keras.models.load_model(head_path)
    model.save(run_dir/"model.keras")
    (run_dir/"classes.json").write_text(json.dumps(CLASSES,ensure_ascii=False,indent=2),encoding="utf-8")
    history=pd.concat([pd.DataFrame(head_history.history).assign(phase="head"),pd.DataFrame(fine_history.history).assign(phase="finetune")],ignore_index=True)
    history.to_csv(run_dir/"history.csv",index=False)
    return model,run_dir

# PASO 2: entrena la CNN sobre los mapas.
model,MODEL_DIR=train_map_model()
print("Modelo guardado en:",MODEL_DIR/"model.keras")

## 5. Evaluación end-to-end interna

Además de la clasificación, esta métrica cuenta como error una letra para la que MediaPipe no detectó mano. Revisa macro F1, matriz de confusión y tasa de detección antes de evaluar fotos propias.

In [ ]:
def evaluate_frame(model,frame,subset_name="train_25k",prefix="internal_test"):
    root=OUTPUT_DIR/subset_name/"maps"
    paths=[str(root/p) for p in frame.map_path.astype(str)]
    labels=frame.label.map(CLASS_TO_ID).to_numpy(dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    def load(path,label):
        image=tf.io.decode_png(tf.io.read_file(path),channels=3)
        image=tf.image.resize(image,(IMAGE_SIZE,IMAGE_SIZE),antialias=True)
        return tf.cast(image,tf.float32),label
    probs=model.predict(ds.map(load).batch(BATCH_SIZE),verbose=0)
    y_true=labels; y_pred=probs.argmax(axis=1)
    detected=frame.detected.astype(bool).to_numpy()
    y_pred[~detected]=CLASS_TO_ID["nothing"]
    metrics={
        "n_images":int(len(frame)),
        "accuracy_end_to_end":float(accuracy_score(y_true,y_pred)),
        "macro_f1_end_to_end":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "hand_detection_rate_on_signs":float(detected[frame.label.to_numpy()!="nothing"].mean()),
        "classification_report":classification_report(y_true,y_pred,labels=list(range(len(CLASSES))),target_names=CLASSES,output_dict=True,zero_division=0),
    }
    (MODEL_DIR/f"{prefix}_metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
    pd.DataFrame({"true":[CLASSES[i] for i in y_true],"prediction":[CLASSES[i] for i in y_pred]}).to_csv(MODEL_DIR/f"{prefix}_predictions.csv",index=False)
    cm=confusion_matrix(y_true,y_pred,labels=list(range(len(CLASSES))),normalize="true")
    fig,ax=plt.subplots(figsize=(12,10))
    ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(ax=ax,xticks_rotation=90,cmap="Blues",colorbar=False,values_format=".2f")
    fig.tight_layout(); fig.savefig(MODEL_DIR/f"{prefix}_confusion.png",dpi=140); plt.show()
    return metrics

internal_metrics=evaluate_frame(model,test_rows)
print({k:v for k,v in internal_metrics.items() if k!="classification_report"})

## 6. Fotos propias (evaluación externa opcional)

Si `data/mis_fotos` tiene una carpeta por clase, se generan sus mapas y se evalúan sin usarlas para entrenar. Mantén esta evaluación aparte del ajuste de hiperparámetros para conservarla como prueba externa.

In [ ]:
# La función de extracción admite cualquier carpeta con subcarpetas de clase.
if OWN_PHOTOS_DIR.is_dir():
    own_df=prepare_maps(OWN_PHOTOS_DIR,"own_photos",per_class=None,progress_every=100,allow_missing=True)
    own_df=own_df[own_df.label.isin(CLASSES)].reset_index(drop=True)
    own_metrics=evaluate_frame(model,own_df,subset_name="own_photos",prefix="own_photos")
    print({k:v for k,v in own_metrics.items() if k!="classification_report"})
else:
    print("No se encontró data/mis_fotos; se omite la prueba externa.")

## 7. Lógica de la sesión Iniciar/Detener

La inferencia actualiza la vista previa todo el tiempo. El buffer de frase solo cambia cuando `recording=True`. `space` agrega un espacio y `nothing` rearma la aceptación para evitar registrar la misma pose en cada fotograma. Para repetir una letra igual, la persona debe pasar por un estado neutral o una pausa breve.

Al detener, la sesión devuelve el texto bruto y genera sugerencias si se proporciona un vocabulario. Esto es corrección ortográfica asistida por diccionario; para generar frases gramaticales se requeriría una capa de lenguaje separada y las sugerencias deben poder aceptarse o rechazarse.

In [ ]:
class SpellingSession:
    def __init__(self, min_confidence=0.80, stable_frames=4):
        self.min_confidence=min_confidence
        self.stable_frames=stable_frames
        self.recording=False
        self.preview=("",0.0)
        self.text=""
        self.candidate=None
        self.streak=0
        self.last_emitted=None

    def start(self):
        self.text=""
        self.recording=True
        self.candidate=None; self.streak=0; self.last_emitted=None

    def observe(self,label,confidence):
        """Llamar en cada frame tras combinar clasificador estático y temporal."""
        self.preview=(label,float(confidence))
        if label is None or confidence < self.min_confidence:
            self.candidate=None; self.streak=0
            if self.recording: self.last_emitted=None
            return self.text
        if label==self.candidate: self.streak+=1
        else: self.candidate=label; self.streak=1
        if not self.recording or self.streak < self.stable_frames:
            return self.text
        if label=="nothing":
            self.last_emitted=None
            return self.text
        if label==self.last_emitted:
            return self.text
        if label=="space":
            if self.text and not self.text.endswith(" "): self.text+=" "
        elif label in STATIC_LETTERS or label in {"J","Z"}:
            self.text+=label
        self.last_emitted=label
        return self.text

    def stop(self):
        self.recording=False
        return self.text.strip()


def suggest_word_corrections(text, vocabulary, max_suggestions=3):
    """Devuelve opciones; nunca reemplaza silenciosamente lo que se deletreó."""
    suggestions=[]
    for word in text.split():
        matches=difflib.get_close_matches(word.lower(),vocabulary,n=max_suggestions,cutoff=0.75)
        if matches and word.lower() not in vocabulary:
            suggestions.append({"word":word,"suggestions":matches})
    return suggestions

# Demo de estado: preview antes de iniciar no modifica el texto.
session=SpellingSession()
session.observe("A",0.97)
print("Preview sin grabar:",session.preview,"| texto:",repr(session.text))
session.start()
for _ in range(4): session.observe("H",0.96)
for _ in range(4): session.observe("O",0.96)
for _ in range(4): session.observe("space",0.93)
for _ in range(4): session.observe("L",0.95)
for _ in range(4): session.observe("A",0.96)
raw_text=session.stop()
print("Texto bruto al detener:",raw_text)
# vocabulario de ejemplo; reemplazar por archivo de español/inglés elegido para el proyecto.
example_vocab={"hola","casa","como","estas","gracias"}
print("Sugerencias:",suggest_word_corrections(raw_text,example_vocab))

## Integración pendiente: cámara y J/Z

- Puede mantenerse el entrenamiento y la preparación de mapas en este notebook de Colab. La cámara de Colab requiere captura desde JavaScript del navegador; `cv2.VideoCapture(0)` no accede directamente a la cámara del computador. Para una interfaz más estable con botones Iniciar/Detener, se puede exportar `model.keras` y ejecutar la inferencia en una app local. Colab incluye un ejemplo oficial de captura de cámara por JavaScript en sus snippets. [Camera Capture de Colab](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb).
- Cuando estén listos los videos, usa el notebook 07 para preparar flujo óptico/entrenar J-Z. La función `SpellingSession.observe` ya puede recibir `J` o `Z` desde esa rama.
- Las imágenes Kaggle etiquetadas J/Z son posturas estáticas y no bastan para entrenar el reconocimiento del movimiento. Tampoco el mapa de un solo fotograma detecta por sí mismo la trayectoria.